# Session 4: Agent Explorer Lab

**Course:** Language Models: ML Basics to Modern AI (BTU Cottbus, M.Sc. AI seminar)
**Session:** 4 of 4
**Lecture reference:** Lecture on agents, tool use, and the ReAct paradigm.

## Learning objectives

By the end of this notebook you should be able to:

- Describe what turns a sequence of LLM calls into an agent and why the autonomy lives in the loop rather than in any single generation.
- Read a ReAct trace (Thought / Action / Observation / Final Answer) and identify which steps are produced by the model and which by the environment.
- Implement a ReAct loop from scratch and dispatch tool calls in it, using a parser that turns model text into structured calls and a registry that maps tool names to Python callables.
- Distinguish prompt-based ReAct from native function calling and from JSON-mode structured output, and pick the appropriate mechanism for a given backend.
- Swap the loop's LLM backend between a deterministic mock, a local small model, and a hosted API without changing the agent code.

The notebook uses a small instruction-tuned model (SmolLM2-135M-Instruct) for the optional local backend and a deterministic mock as the default so the loop always runs end to end.


## §1 Primer: programs as agents, ReAct, and tool calling

This primer introduces the three ideas behind the §5 build: what makes a program an agent in the first place, the ReAct paradigm that structures the loop, and the three closely related mechanisms that let a model call code (prompt-based tool use, native function calling, and structured output). Read it before the §3 demo so the traces have something to anchor to.

### §1a What makes a program an agent

Picture two ways of asking a language model to multiply 17 by 23 and add 4. The first hands the model the question and reads back whatever string it produces. If the model gets it wrong, you have a wrong answer; there is no second attempt and no way for the model to check its work. The second wraps the model call in a small loop. The model writes a step, the program runs that step (here, evaluating arithmetic), the program shows the model the result, and the model writes the next step. The model is the same; the second arrangement is what people call an agent. The autonomy lives in the loop, not in any single generation.

The intuition is that a one-shot generation has no way to recover from its own mistakes. The model sees the question and emits a continuation, and whatever it emits is the final word. A loop changes that because each iteration produces a fresh prompt that includes everything the model has said and seen so far. If the model called a tool and the tool returned an error, the next prompt contains that error, and the model has the chance to try a different approach. If the model wrote nonsense, the dispatcher can flag it and the next prompt asks for a corrected step. The looping gives the system room to be wrong on a single step without being wrong overall.

A working definition: an agent is a program that combines an LLM with a loop over external actions, where each iteration appends the action's result to the model's input. The LLM provides the policy (what to do next given what has happened so far). The loop provides the persistence (the memory of past actions and observations) and the side effects (the actual calls into the environment). Take either piece away and the result stops being agentic. An LLM with no loop is a one-shot generator. A loop with no LLM is a fixed-script program.

### §1b The ReAct paradigm

ReAct is the most widely used loop format for prompt-based agents. The acronym names the three roles each iteration cycles through: the model writes a Thought (a short natural-language sentence about what it plans to do), then an Action (a tool call expressed in a fixed syntax), then receives an Observation (the tool's return value, provided by the program rather than the model). The loop repeats until the model writes a Final Answer instead of another Action, at which point the dispatcher returns the answer and stops.

Four properties make this framing useful. First, the reasoning is externalised: the model's plan is written into the transcript as text, so the next iteration can see what was tried and why. Second, the memory is persistent: every Thought, Action, and Observation accumulates in the prompt the model sees on the next step, so context grows across the run. Third, the tool interface is explicit: the model has to express its calls in a syntax the parser recognises, which makes the boundary between model and environment visible. Fourth, the structure is parseable: a regex over the completion text is enough to extract Action lines, so no special API support is required.

The standard system-prompt layout follows the same shape as the loop. Describe the tools (name, argument type, return type, when to use). Show the model the Thought / Action / Observation format with one worked example. Tell the model to end with `Final Answer: <answer>` when it is ready. The §5 build uses exactly this layout.

### §1c Tool calling, function calling, and structured output

Three terms get used near each other in the agent literature and they refer to related but distinct mechanisms. Sorting them out helps when you read an API reference or pick a backend for a real system.

**Tool calling via prompt (what the §5 build implements).** The model is asked to follow the ReAct format in plain text. The dispatcher parses the model's output with a regex, runs the tool, and feeds the result back in the next prompt. This works on any text-completion API because no special model capability is required. It also degrades clearly: if the model gets the format wrong, the parser fails and the dispatcher can ask the model to retry. The cost is reliability; small models or models without instruction tuning often produce slightly malformed Action lines that the parser misses.

**Native function calling (Groq, OpenAI, Anthropic, and similar APIs).** The same idea, but the model is trained to emit tool calls as structured JSON tokens that the API surfaces as a separate field on the response. The application provides a JSON schema for each available tool; the API returns either a normal text completion or a structured tool-call object the application can dispatch directly. This is the production-grade path because the model is trained on the exact format, so parse failures are rare and the application code is shorter.

**Structured output, or JSON mode.** The lower-level primitive both of the above build on. The model is constrained to produce output matching a JSON schema. Function calling is structured output where the schema describes a tool invocation; an application using structured output for a non-tool purpose (extracting fields from a document, returning a typed result) uses the same machinery for a different shape.

When to use which: prompt-based ReAct is the right choice for teaching and for any backend that does not support native function calling, since it works everywhere. Native function calling is the right choice for any production system on a backend that supports it, because the reliability gain is substantial. Structured output without a tool layer is the right choice when the model is producing a typed result for downstream code to consume rather than calling out to a tool.


## §2 Setup

The cell below loads the local SmolLM2-Instruct model and defines an `LLMCaller` abstraction with three backends: a deterministic mock (the default; returns pre-recorded ReAct steps so the loop runs end to end without external dependencies), a local SmolLM2 call (free but unreliable for tool use at 135M parameters), and a Groq API call (the production-quality path; requires `GROQ_API_KEY` in the environment). The agent loop in §5 works against any backend that follows the caller contract, so swapping backends never touches the loop code.


In [ ]:
"""§2 Setup: imports, seed, device, local LLM, LLMCaller abstraction."""

import os
import re
import textwrap
from dataclasses import dataclass
from pathlib import Path

import torch
from IPython.display import HTML, display
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 0
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}, torch: {torch.__version__}")

# Local model load (used by LocalCaller below). Cached from Session 3.
INSTRUCT_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
local_tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_NAME)
local_model = AutoModelForCausalLM.from_pretrained(INSTRUCT_NAME).to(DEVICE).eval()
print(f"local LLM: {INSTRUCT_NAME} loaded.")


# An LLMCaller is any callable that takes (system_prompt: str, user_prompt: str)
# and returns the model's text completion as a string. The agent loop in §5
# works against any caller that follows this contract.

def make_mock_caller():
    """Deterministic mock that returns pre-recorded ReAct steps keyed by the
    user prompt's trailing question. Used by default so the notebook runs
    end-to-end with no external dependencies.
    """
    SCRIPTED = {
        "What is 17 * 23 + 4?": [
            "Thought: I need to compute an arithmetic expression. Use the calculator.\n"
            "Action: calculator(17 * 23 + 4)\n",
            "Thought: The calculator returned 395. That is the answer.\n"
            "Final Answer: 395\n",
        ],
        "What is the capital of the Solomon Islands?": [
            "Thought: I do not know this off the top of my head. Use the search tool.\n"
            "Action: search(capital of the Solomon Islands)\n",
            "Thought: The search returned 'Honiara'. That is the capital.\n"
            "Final Answer: Honiara\n",
        ],
    }

    def _caller(system_prompt: str, user_prompt: str) -> str:
        # Find the most recent question line in user_prompt.
        for question, steps in SCRIPTED.items():
            if question in user_prompt:
                # Count how many Observations are already in the transcript;
                # that tells us which scripted step to return.
                step_idx = user_prompt.count("Observation:")
                if step_idx < len(steps):
                    return steps[step_idx]
                return "Final Answer: (mock ran out of scripted steps)\n"
        return "Final Answer: (mock has no script for this question)\n"

    return _caller


def make_local_caller(temperature: float = 0.3, max_new_tokens: int = 160):
    """Use the local SmolLM2-Instruct model. Free but unreliable for tool use
    because of the small parameter count. Useful for demonstration; not the
    default."""
    def _caller(system_prompt: str, user_prompt: str) -> str:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        text = local_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        ids = local_tokenizer.encode(text, return_tensors="pt").to(DEVICE)
        attention_mask = torch.ones_like(ids)
        with torch.no_grad():
            out = local_model.generate(
                ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=0.9,
                pad_token_id=local_tokenizer.eos_token_id,
            )
        return local_tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

    return _caller


def make_groq_caller(model_name: str = "llama-3.1-8b-instant", temperature: float = 0.1):
    """Use the Groq API. Requires GROQ_API_KEY in the environment. Fast and
    reliable; this is the production-quality path."""
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        return None
    from groq import Groq
    client = Groq(api_key=api_key)

    def _caller(system_prompt: str, user_prompt: str) -> str:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=temperature,
            max_tokens=300,
            stop=["Observation:"],
        )
        return response.choices[0].message.content

    return _caller


# Default caller used by the agent loop in §5. Override by reassigning
# `default_caller` to make_local_caller() or make_groq_caller() before running
# the §5 example.
default_caller = make_mock_caller()
print("Default LLM caller: deterministic mock (use make_local_caller() or "
      "make_groq_caller() to switch).")


## §3 Guided exploration: three pre-recorded ReAct traces

The cell below walks through three pre-recorded ReAct traces produced by a stronger model (llama-3.1-8b-instant via the Groq API) on the same two example queries you will use in §5, plus a third combined query that requires both tools. Reading these before you build the loop in §5 makes the loop dynamics visible: each block is a small dialogue in which the agent writes a Thought and an Action, the environment supplies an Observation, and the loop continues until the agent writes a Final Answer. The traces are rendered as static HTML; no controls, no live model calls.


In [ ]:
"""§3 Three pre-recorded ReAct traces from a stronger model.

These traces were produced by llama-3.1-8b-instant via the Groq API on the
same two example queries used in §5, plus a third combined query. The point
of showing them here is to make the loop dynamics visible before you build
the loop. Each block reads like a small dialogue between the agent (Thought,
Action) and the environment (Observation), terminating in Final Answer.
"""

TRACES = [
    {
        "question": "What is 17 * 23 + 4?",
        "steps": [
            ("Thought", "I need to compute an arithmetic expression with multiplication "
                        "and addition. The calculator tool can handle this directly."),
            ("Action", "calculator(17 * 23 + 4)"),
            ("Observation", "395"),
            ("Thought", "The calculator returned 395. That is the final answer."),
            ("Final Answer", "395"),
        ],
    },
    {
        "question": "What is the capital of the Solomon Islands?",
        "steps": [
            ("Thought", "I do not have a confident memory of this. Use the search tool."),
            ("Action", "search(capital of the Solomon Islands)"),
            ("Observation", "Honiara is the capital and largest city of the Solomon Islands."),
            ("Thought", "Honiara is the answer."),
            ("Final Answer", "Honiara"),
        ],
    },
    {
        "question": "If a flight from Berlin to Honiara takes 30 hours total and I want to "
                    "leave next Monday at 09:00 local Berlin time, what is the local arrival "
                    "time in the Solomon Islands?",
        "steps": [
            ("Thought", "I need the time zone offset of Honiara, then to add 30 hours, then "
                        "to adjust for the offset. Search first for the time zone."),
            ("Action", "search(Honiara time zone)"),
            ("Observation", "Honiara uses Solomon Islands Time, UTC+11. Berlin in winter "
                            "is UTC+1, so the offset is +10 hours."),
            ("Thought", "Departure 09:00 Berlin time. Add 30 hours flight, add 10 hours "
                        "offset. Use calculator: 9 + 30 + 10 = 49, modulo 24 is 1."),
            ("Action", "calculator((9 + 30 + 10) % 24)"),
            ("Observation", "1"),
            ("Thought", "Arrival is at 01:00 local Honiara time. Departure Monday plus "
                        "30 hours plus 10 hour offset lands on early Wednesday morning."),
            ("Final Answer", "01:00 local Honiara time on Wednesday."),
        ],
    },
]


def _render_trace(trace: dict) -> str:
    colors = {
        "Thought": "#1e293b",
        "Action": "#0369a1",
        "Observation": "#15803d",
        "Final Answer": "#b91c1c",
    }
    rows = "".join(
        f"<tr><td style='padding:4px 12px;color:{colors[role]};font-weight:600;"
        f"white-space:nowrap;vertical-align:top;width:160px;'>{role}:</td>"
        f"<td style='padding:4px 12px;font-family:Georgia,serif;line-height:1.4;'>"
        f"{content}</td></tr>"
        for role, content in trace["steps"]
    )
    return (
        f"<p style='margin:14px 0 4px;font-weight:700;'>Question: {trace['question']}</p>"
        f"<table style='border-collapse:collapse;border:1px solid #cbd5e1;width:100%;'>"
        f"{rows}</table>"
    )


for trace in TRACES:
    display(HTML(_render_trace(trace)))


## §4 Warm-ups

Two short exercises before the deep build. Warm-up 1 parses an Action line out of an LLM completion using a regex; the §5 loop relies on a more permissive version of the same parser. Warm-up 2 implements the calculator tool the agent will call, with a safe AST-based evaluator that refuses anything that is not pure arithmetic.


In [ ]:
"""§4 Warm-up 1 (exercise): parse an Action line from an LLM completion.

Implement `parse_action(line)` so that given a string of the form
'Action: tool_name(argument)' it returns the tuple (tool_name, argument) with
whitespace stripped around both. Empty arguments such as 'Action: noop()' should
return ('noop', ''). If the line does not contain that pattern, return
(None, None) so the caller can detect the failure.
"""

import re


def parse_action(line: str) -> tuple[str | None, str | None]:
    """Parse a string like 'Action: tool_name(argument)' into (name, argument).
    Returns (None, None) if the line does not match the pattern."""
    # TODO: use a regex to extract the tool name and the argument from a string
    # of the form 'Action: name(arg)'. Strip whitespace around both. If the
    # input does not contain that pattern, return (None, None).
    raise NotImplementedError


# Smoke tests (wrapped so the notebook still runs before the stub is filled in).
try:
    assert parse_action("Action: calculator(2 + 2 * 7)") == ("calculator", "2 + 2 * 7")
    assert parse_action("Action: search(capital of the Solomon Islands)") == ("search", "capital of the Solomon Islands")
    assert parse_action("Action: noop()") == ("noop", "")
    assert parse_action("not an action line") == (None, None)
    print("parse_action: 4/4 smoke tests pass")
except NotImplementedError:
    print("parse_action not implemented yet.")
except AssertionError:
    print("parse_action implemented but at least one smoke test failed; review the cases above.")


In [ ]:
"""§4 Warm-up 2 (exercise): a small safe arithmetic evaluator.

Implement `calculator(expression)` so that it parses the input as a Python
expression and evaluates only the safe arithmetic subset: integer and float
literals, the binary operators + - * / % and **, unary plus and minus, and
parentheses. Anything else (attribute access, function calls, names) must be
rejected. Catch SyntaxError, ValueError, and ZeroDivisionError at the top
level and return a short 'error: ...' string instead of raising, so the agent
loop can hand the error message back to the LLM as an Observation.
"""

import ast
import operator


_ALLOWED_BINOPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
}
_ALLOWED_UNARYOPS = {
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def calculator(expression: str) -> str:
    """Safely evaluate an arithmetic expression. Allowed: + - * / % **, unary minus,
    parentheses, integer and float literals. Anything else raises ValueError."""
    # TODO: parse the expression with ast.parse in 'eval' mode, then walk the
    # AST and evaluate only nodes that are constants, BinOp with the allowed
    # operators, or UnaryOp with the allowed operators. Reject everything else
    # by raising ValueError. Catch SyntaxError, ValueError, and ZeroDivisionError
    # at the top level and return a short error string instead of raising.
    raise NotImplementedError


# Smoke tests (wrapped so the notebook still runs before the stub is filled in).
try:
    assert calculator("17 * 23 + 4") == "395"
    assert calculator("(9 + 30 + 10) % 24") == "1"
    assert calculator("2 ** 10") == "1024"
    assert calculator("__import__('os').system('ls')").startswith("error:")
    print("calculator: 4/4 smoke tests pass")
except NotImplementedError:
    print("calculator not implemented yet.")
except AssertionError:
    print("calculator implemented but at least one smoke test failed; review the cases above.")


## §5 Deep build: implement the ReAct loop from scratch

Five subtasks build the agent. Subtask 1 registers the two tools (the calculator from Warm-up 2 plus a small search stub). Subtask 2 writes the system prompt that teaches the model the ReAct format. Subtask 3 is a more permissive Action parser than Warm-up 1, handling three common call syntaxes. Subtask 4 is the loop itself, which alternates LLM calls with tool dispatch and accumulates a transcript. Subtask 5 runs the agent on two example queries and prints the traces.

The default `LLMCaller` is the deterministic mock from §2, so the loop runs to completion on Colab with no API key or local model. To swap in the local SmolLM2 backend or the Groq backend, reassign `default_caller` to `make_local_caller()` or `make_groq_caller()` before re-running Subtask 5.


In [ ]:
"""§5 Subtask 1 (exercise): the tool registry.

SEARCH_CORPUS and the search() stub below are provided. Your job is to
assemble the TOOLS dict that the agent dispatcher in Subtask 4 will look up by
name. The convention is that every tool is a callable that takes a single
string argument and returns a string. The calculator from Warm-up 2 already
satisfies that contract.
"""

# search() returns canned text for a handful of queries. In production this
# would be a real retrieval call (BM25, dense embedding, web search).
SEARCH_CORPUS = {
    "capital of the Solomon Islands":
        "Honiara is the capital and largest city of the Solomon Islands.",
    "Honiara time zone":
        "Honiara uses Solomon Islands Time, UTC+11. Berlin in winter is UTC+1.",
    "Maria Henson":
        "Maria Henson is the co-founder and CEO of BlueQuokka, Inc.",
    "BlueQuokka":
        "BlueQuokka, Inc. is an Australian-German company that builds rugged "
        "field-research tablets, founded in 2017.",
}


def search(query: str) -> str:
    q = query.strip().lower()
    for key, value in SEARCH_CORPUS.items():
        if key.lower() in q or q in key.lower():
            return value
    return "(no result; the search tool only knows a handful of queries in this lab)"


# TODO: build a TOOLS dict mapping the string 'calculator' to the calculator
# function from Warm-up 2, and 'search' to the search function above. The
# agent dispatcher in Subtask 4 looks tools up in this dict by name.
TOOLS = {}  # replace

print("Tools registered:", list(TOOLS))
if TOOLS:
    try:
        print(f"calculator('3 + 4 * 5') -> {TOOLS['calculator']('3 + 4 * 5')}")
    except (KeyError, NotImplementedError) as err:
        print(f"calculator not yet wired up: {type(err).__name__}")
    try:
        print(f"search('capital of the Solomon Islands') -> {TOOLS['search']('capital of the Solomon Islands')[:60]}...")
    except (KeyError, NotImplementedError) as err:
        print(f"search not yet wired up: {type(err).__name__}")
else:
    print("TOOLS is empty; downstream cells will skip until you populate it.")


In [ ]:
"""§5 Subtask 2: the system prompt that teaches the model the ReAct format."""

SYSTEM_PROMPT = textwrap.dedent('''\
    You are a helpful assistant that answers questions by reasoning step by
    step and calling tools when useful. You have access to two tools:

      calculator(expression): evaluates an arithmetic expression and returns
        the result as a string. Use this for any arithmetic the user asks
        about.

      search(query): looks up a fact in a small knowledge base and returns a
        short passage as a string. Use this for facts you are not sure about.

    Respond in the following format, one block at a time:

      Thought: <a short sentence about what you need to do next>
      Action: <tool_name>(<argument>)

    After each Action, you will receive an Observation containing the tool's
    return value. Use it to plan the next step. When you are ready to answer,
    finish with:

      Final Answer: <your final answer>

    Worked example:

      Question: What is 12 * 7 minus 5?
      Thought: I should compute the arithmetic with the calculator.
      Action: calculator(12 * 7 - 5)
      Observation: 79
      Thought: The calculator returned 79.
      Final Answer: 79

    Do not invent tools that are not listed above. Do not output the
    Observation line yourself; that comes from the environment.
''')

print(SYSTEM_PROMPT)


In [ ]:
"""§5 Subtask 3 (exercise): a more permissive Action parser used by the loop.

Implement `parse_action_robust(text)` that scans `text` line by line, finds
the first line starting with 'Action:', and parses the body in one of three
formats so the loop tolerates small format drift from the model:
  Action: tool(arg)
  Action: tool[arg]
  Action: tool: arg
Return (tool, arg) with whitespace stripped on both. If no Action line is
found or none of the formats match, return (None, None) so the loop can
surface the parse failure to the model.
"""

import re


def parse_action_robust(text: str) -> tuple[str | None, str | None]:
    """Find the first Action line in `text` and parse it into (tool, arg).
    Returns (None, None) if no Action line is found or none of the formats match."""
    # TODO: split the input into lines and walk them. Skip any line that does
    # not start with 'Action:'. Strip the prefix and try three regex patterns
    # in order: parentheses, brackets, then a 'name: rest-of-line' fallback.
    # Return the first match as (tool, arg). If no Action line ever matches,
    # return (None, None).
    raise NotImplementedError


# Smoke tests (wrapped so the notebook still runs before the stub is filled in).
try:
    assert parse_action_robust("Action: calculator(17 * 23)") == ("calculator", "17 * 23")
    assert parse_action_robust("Action: search[Solomon Islands]") == ("search", "Solomon Islands")
    assert parse_action_robust("Action: search: Honiara time zone") == ("search", "Honiara time zone")
    assert parse_action_robust("Thought: nothing to do here.") == (None, None)
    print("parse_action_robust: 4/4 smoke tests pass")
except NotImplementedError:
    print("parse_action_robust not implemented yet.")
except AssertionError:
    print("parse_action_robust implemented but at least one smoke test failed; review the cases above.")


In [ ]:
"""§5 Subtask 4 (exercise): the ReAct loop.

The render_transcript helper is provided. Your job is to implement run_react,
which alternates LLM calls with tool dispatch until the model writes a Final
Answer or the step budget runs out.
"""


def render_transcript(question: str, steps: list[dict]) -> str:
    """Format a question and an alternating list of (thought/action/observation)
    steps as the prompt the LLM sees on the next iteration."""
    lines = [f"Question: {question}"]
    for step in steps:
        if "thought" in step:
            lines.append(f"Thought: {step['thought']}")
        if "action" in step:
            lines.append(f"Action: {step['action']}")
        if "observation" in step:
            lines.append(f"Observation: {step['observation']}")
    return "\n".join(lines)


def run_react(question, llm, system_prompt, tools, max_steps=6):
    """Run the ReAct loop until Final Answer or max_steps. Returns a dict with
    keys 'answer' (str or None), 'steps' (int), and 'trace' (list of role,
    content pairs suitable for rendering)."""
    # TODO: maintain a trace as a list of (role, content) pairs. For each step
    # up to max_steps:
    #   - build the user prompt by joining 'Question: <question>' with the
    #     trace lines so far (one 'Role: content' line per trace entry).
    #   - call llm(system_prompt, user_prompt) to get a completion.
    #   - look for the first Thought line and append it to the trace.
    #   - look for a Final Answer line; if found, append and return the dict.
    #   - otherwise call parse_action_robust on the completion. If it returns
    #     (None, None), append an Error and return.
    #   - look the tool up in `tools`; if missing, set observation to an error
    #     string. Otherwise call the tool; if it raises, format the exception
    #     into an error string.
    #   - append the Action and Observation to the trace and continue.
    # If max_steps elapses without a Final Answer, append an Error and return.
    raise NotImplementedError


# Smoke test against the mock caller (wrapped so an unimplemented stub does
# not crash the notebook).
try:
    result = run_react(
        "What is 17 * 23 + 4?",
        default_caller,
        SYSTEM_PROMPT,
        TOOLS,
        max_steps=4,
    )
    print(f"answer: {result['answer']}, steps: {result['steps']}")
    for role, content in result["trace"]:
        print(f"  {role}: {content}")
except NotImplementedError:
    print("run_react not implemented yet.")
except (NameError, TypeError) as err:
    print(f"run_react could not run: {type(err).__name__}: {err}")


In [ ]:
"""§5 Subtask 5 (exercise): run the agent on two example queries.

The HTML helper and the QUERIES list are provided. Your job is to call
run_react on each question with the default_caller, SYSTEM_PROMPT, and TOOLS,
then display the trace using _render_run and also print it in plain text.
"""


def _render_run(question: str, result: dict) -> str:
    colors = {
        "Thought": "#1e293b",
        "Action": "#0369a1",
        "Observation": "#15803d",
        "Final Answer": "#b91c1c",
        "Error": "#7c2d12",
    }
    rows = "".join(
        f"<tr><td style='padding:4px 12px;color:{colors.get(role, '#475569')};"
        f"font-weight:600;white-space:nowrap;vertical-align:top;width:160px;'>{role}:</td>"
        f"<td style='padding:4px 12px;font-family:Georgia,serif;line-height:1.4;'>{content}</td></tr>"
        for role, content in result["trace"]
    )
    summary = (
        f"<p style='margin:4px 0 8px;color:#475569;font-size:13px;'>"
        f"Final answer: <strong>{result['answer']}</strong> in {result['steps']} step(s)."
        f"</p>"
    )
    return (
        f"<p style='margin:14px 0 4px;font-weight:700;'>Question: {question}</p>"
        f"<table style='border-collapse:collapse;border:1px solid #cbd5e1;width:100%;'>"
        f"{rows}</table>"
        f"{summary}"
    )


QUERIES = [
    "What is 17 * 23 + 4?",
    "What is the capital of the Solomon Islands?",
]


# Guard against unimplemented run_react / parse_action_robust / TOOLS: probe
# once on the first question; if that raises, skip the loop with a clear
# message so the notebook still runs end-to-end.
_loop_ready = True
try:
    _probe = run_react(QUERIES[0], default_caller, SYSTEM_PROMPT, TOOLS, max_steps=2)
except (NotImplementedError, NameError, TypeError, KeyError):
    _loop_ready = False

if not _loop_ready:
    print("Skipping the run: complete Subtasks 1, 3, and 4 first so TOOLS, "
          "parse_action_robust, and run_react are all wired up.")
else:
    for question in QUERIES:
        # TODO: call run_react on the question with default_caller,
        # SYSTEM_PROMPT, TOOLS, and a max_steps budget. Display the resulting
        # trace with _render_run, then print a plain-text trace so the
        # comparison shows up outside the HTML renderer too.
        pass

# To try the local SmolLM2 backend (free but unreliable with 135M parameters):
#     local_caller = make_local_caller()
#     run_react("What is 17 * 23 + 4?", local_caller, SYSTEM_PROMPT, TOOLS)
# To try Groq (requires GROQ_API_KEY in your environment):
#     groq_caller = make_groq_caller()
#     if groq_caller:
#         run_react("What is 17 * 23 + 4?", groq_caller, SYSTEM_PROMPT, TOOLS)


## §6 Recap

An agent is a loop with three pieces: a prompt format that lets the model express tool calls (here, the ReAct Thought / Action / Observation / Final Answer structure), a parser that turns the model's text into structured calls (here, `parse_action_robust`), and a dispatcher that runs the call against a tool registry and feeds the result back into the next prompt (here, `run_react`). Production agent frameworks (LangChain, LlamaIndex, the OpenAI Agents SDK) elaborate the same three pieces with richer prompt templates, native function-calling backends, persistent memory stores, and multi-agent orchestration; the core loop is what you wrote in Subtask 4.
